In [30]:
import io
import json
import os

import open_clip
import pandas as pd
import torch
from PIL import Image
from matplotlib import pyplot as plt
from torch.utils.data.dataloader import DataLoader

from homework_3.functions import parse_img, api_response_to_tensor

In [31]:
class ClipModel(torch.nn.Module):
    def __init__(self, model_name="ViT-B-32", pretrained="laion2b_s34b_b79k"):
        super().__init__()
        self.clip_backbone, _, self.preprocess = open_clip.create_model_and_transforms(model_name=model_name,
                                                                                       pretrained=pretrained)
        for p in self.clip_backbone.parameters():
            p.requires_grad = False
        self.clip_backbone.eval()
        self.tokenizer = open_clip.get_tokenizer('ViT-B-32')

        self.class_labels = ["class1", "class2", "class3", "class4"]
        self.class_tokens = self.tokenizer(self.class_labels)
        self.class_features = self.clip_backbone.encode_text(self.class_tokens)
        if torch.cuda.is_available():
            self.class_features /= self.class_features.norm(dim=-1, keepdim=True)
            self.class_features = self.class_features.cuda()

        self.head = torch.nn.Linear(self.clip_backbone.visual.output_dim,
                                    self.clip_backbone.visual.output_dim)
        if torch.cuda.is_available():
            self.clip_backbone.cuda()
            self.head.cuda()

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            image_features = self.clip_backbone.encode_image(images)
        image_features = self.head(image_features)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        class_similarity = (100.0 * image_features @ self.class_features.T).softmax(dim=-1)
        return class_similarity


In [35]:
class ApiResponsesDataset(torch.utils.data.Dataset):
    def __init__(self, clip_preprocess, img_dir="../img", img_series=None, responses_cache="../cache/responses.json"):
        super().__init__()

        self.clip_preprocess = clip_preprocess
        self.responses_json = json.load(open(responses_cache, "r"))

        self.raw_images = [Image.open(f"{img_dir}/{p}") for p in os.listdir(img_dir)]
        self.raw_images.extend([Image.open(io.BytesIO(i['bytes'])) for i in img_series])

        self.data = []
        self.labels = []

        self._preprocess()

    def _preprocess(self):
        self.data = [self.clip_preprocess(raw_img) for raw_img in self.raw_images]

        try:
            for raw_img in self.raw_images:
                _, img_hash = parse_img(raw_img)
                response = self.responses_json[img_hash]
                label = api_response_to_tensor(response)
                self.labels.append(label)
        except KeyError:
            exit(f"Dataset integrity error. Not all images have responses.")

        assert len(self.data) == len(self.labels), "Unequal amount of data and labels loaded"


    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index], self.labels[index]

In [33]:
# TODO Add validation
def train_model(model, optimizer, criterion, train_data, num_epochs):
    losses = []
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        epoch_loss = 0.0
        for data, labels in train_data:
            if torch.cuda.is_available():
                data = data.cuda()
                labels = labels.cuda()
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(train_data))
        print(f"Train loss: {losses[-1]}")

    plt.plot(losses)
    plt.title("Loss curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

    return model



In [36]:
WEIGHTS_DIR = "../weights"
RANDOM_SEED = 161
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
NAME = f"{NUM_EPOCHS}_{BATCH_SIZE}"

torch.random.manual_seed(RANDOM_SEED)

clip_model = ClipModel()

df = pd.read_parquet("../datasets/cifar100_train.parquet")
series = df.img[:100]

dataset = ApiResponsesDataset(clip_model.preprocess, img_series=series)
train_dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

adam_optimizer = torch.optim.Adam(clip_model.head.parameters(), lr=LEARNING_RATE)

crossEntropyLoss = torch.nn.CrossEntropyLoss()

In [37]:
trained_model = train_model(model=clip_model,
                            optimizer=adam_optimizer,
                            criterion=crossEntropyLoss,
                            train_data=train_dataloader,
                            num_epochs=NUM_EPOCHS)

torch.save(trained_model.head.state_dict(), f"{WEIGHTS_DIR}/{NAME}.pth")


107
107
Epoch 1/100
Train loss: 2.642300014142637
Epoch 2/100
Train loss: 2.5912374213889793
Epoch 3/100
Train loss: 2.579872736224422
Epoch 4/100
Train loss: 2.5660926368501453
Epoch 5/100
Train loss: 2.559726768069797
Epoch 6/100
Train loss: 2.5519556116174766
Epoch 7/100
Train loss: 2.5474569885819047
Epoch 8/100
Train loss: 2.5437222056918674
Epoch 9/100
Train loss: 2.5420901951966464
Epoch 10/100
Train loss: 2.5405755881909973
Epoch 11/100
Train loss: 2.536714063750373
Epoch 12/100
Train loss: 2.535810708999634
Epoch 13/100
Train loss: 2.5319319698545666
Epoch 14/100
Train loss: 2.5318200632377907
Epoch 15/100
Train loss: 2.531449622578091
Epoch 16/100
Train loss: 2.532654303091544
Epoch 17/100
Train loss: 2.531049688657125
Epoch 18/100
Train loss: 2.533866454053808
Epoch 19/100
Train loss: 2.535537189907498
Epoch 20/100
Train loss: 2.5351663872047707
Epoch 21/100
Train loss: 2.5356685806203774
Epoch 22/100
Train loss: 2.532746054508068
Epoch 23/100
Train loss: 2.5321489925737732


KeyboardInterrupt: 